# PDF Question Answering (RAG)

A small RAG pipeline I built to ask questions about a PDF.

Steps: load the PDF -> split it into chunks -> embed the chunks -> store them in a
vector database -> retrieve the relevant ones for a question -> send them to Gemini
as context.

The model is told to answer only from the document, so anything not in the PDF gets
a "could not find it" instead of a made-up answer.

Made to run in Google Colab. You need a free Gemini API key from
https://aistudio.google.com/apikey

## Upload the PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
# put the name of the file you uploaded here
PDF_FILE = "your_file.pdf"

In [ ]:
loader = PyPDFLoader(PDF_FILE)
documents = loader.load()

In [ ]:
print(len(documents))

print(documents[0].page_content[:1000])

print(documents[0].metadata)

## Split into chunks

A full page is too big to retrieve well, so I split the text into 1000 character
chunks. The 200 character overlap makes sure a sentence sitting on a chunk boundary
still shows up complete in one of them.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

In [ ]:
print("Number of chunks:", len(chunks))

print(chunks[0].page_content)

print(chunks[0].metadata)

## Embeddings

MiniLM turns each chunk into a 384 dimension vector. It runs locally so there is no
API cost for this part.

In [ ]:
from langchain_chroma import Chroma

In [ ]:
!pip install -q sentence-transformers

In [ ]:
!pip install -q langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
sample_text = chunks[0].page_content

vector = embeddings.embed_query(sample_text)

print("Text:")
print(sample_text)

print("\nEmbedding length:", len(vector))

print("\nFirst 10 values:")
print(vector[:10])

## Store the chunks in Chroma

Chroma embeds every chunk and indexes it so we can search by meaning instead of by
keyword.

In [ ]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [ ]:
print("Number of stored chunks:", vectorstore._collection.count())

## Try a search first

Before adding the LLM I wanted to check that the retrieval part works on its own. If
the chunks coming back here are wrong, the final answer will be wrong too.

Change the question to something that is actually in your PDF.

In [ ]:
question = "What is this document about?"

In [ ]:
results = vectorstore.similarity_search(
    question,
    k=3
)

In [ ]:
for i, result in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(result.page_content)
    print("\nMetadata:", result.metadata)

## Connect Gemini

Testing the API key works before plugging it into the chain.

In [ ]:
!pip install -q google-genai

In [ ]:
from google import genai

In [ ]:
from getpass import getpass

GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Explain what RAG is in one sentence."
)

print(response.text)

## Retriever

Same similarity search as before, just wrapped in the retriever interface so it can
be used inside a chain.

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [ ]:
docs = retriever.invoke(question)

for i, doc in enumerate(docs):
    print(f"\n--- Document {i+1} ---")
    print(doc.page_content)

## Prompt

The important line is the last one. Without it the model answers from its own
knowledge instead of the document.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template("""
Answer the question using only the context provided below.

Context:
{context}

Question:
{question}

If the answer is not present in the context, say that you
could not find the answer in the document.
""")

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

## Build the chain

The question goes to the retriever, the retrieved docs become `context`, and
`RunnablePassthrough` passes the question through to `question`. Both get filled
into the prompt.

Checking the filled prompt first before adding the model.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt_template
)

In [ ]:
test_prompt = rag_chain.invoke(question)

print(test_prompt)

In [ ]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=test_prompt.to_string()
)

print(response.text)

## Same thing with the LangChain wrapper

Using `ChatGoogleGenerativeAI` instead of the raw client so the model can sit inside
the chain.

`StrOutputParser` at the end matters: without it `.invoke()` gives back a message
object and printing it dumps a huge block of metadata instead of the answer.

In [ ]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

gemini_llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

In [ ]:
response = (gemini_llm | StrOutputParser()).invoke(
    "Explain what RAG is in one sentence."
)

print(response)

In [ ]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt_template
    | gemini_llm
    | StrOutputParser()
)

## Testing

In [ ]:
response = rag_chain.invoke(question)

print(response)

In [ ]:
response = rag_chain.invoke(
    "Summarise the main points of the first section."
)

print(response)

### Question that is not in the PDF

This is the test that actually matters. Gemini knows this answer from its own
training, so if the pipeline is working it should still refuse and say it could not
find it in the document.

In [ ]:
response = rag_chain.invoke(
    "What is the current population of India?"
)

print(response)